In [12]:
!pip install numpy pandas scipy scikit-learn pyod numba

You should consider upgrading via the '/home/mhussein/.pyenv/versions/3.9.0/bin/python3.9 -m pip install --upgrade pip' command.


In [13]:
!git clone https://github.com/winstonll/COPOD.git

fatal: destination path 'COPOD' already exists and is not an empty directory.


In [14]:
import os

os.listdir("COPOD")

['.git', 'README.md', '.gitattributes', 'data', 'models', 'results']

In [15]:
import os

os.listdir("COPOD/data")

['breastw.mat',
 'shuttle.mat',
 'wine.mat',
 'mammography.mat',
 'pima.mat',
 'literature',
 'cardio.mat',
 'satimage-2.mat',
 'satellite.mat',
 'optdigits.mat',
 'semantic',
 'ionosphere.mat',
 'wbc.mat',
 'letter.mat',
 'arrhythmia.mat',
 'lympho.mat',
 'speech.mat']

In [16]:
from scipy.io import loadmat
import os

data_path = "COPOD/data/breastw.mat"

mat = loadmat(data_path)

X = mat["X"]
y = mat["y"].ravel()

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Number of outliers:", y.sum())

X shape: (683, 9)
y shape: (683,)
Number of outliers: 239


In [17]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.4,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

Training set: (409, 9)
Test set: (274, 9)


In [20]:
import sys
sys.path.append("COPOD")

from models.cod import COPOD

ModuleNotFoundError: No module named 'pyod'

In [21]:
import sys
!{sys.executable} -m pip install pyod


  Obtaining dependency information for pyod from https://files.pythonhosted.org/packages/0c/52/ba08231c789ce9585d627c789c98e0792ee288ade6975f9d834d324f5369/pyod-3.6.7-py3-none-any.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 kB 534.9 kB/s eta 0:00:00MB/s eta 0:00:01
  Obtaining dependency information for joblib>=1.5 from https://files.pythonhosted.org/packages/18/53/84099323c2ec4be98d935f63c033ac4151ee83836ca1050ede3b3aadf155/joblib-1.6.0-py3-none-any.whl.metadata
  Obtaining dependency information for cloudpickle>=3.0 from https://files.pythonhosted.org/packages/88/39/799be3f2f0f38cc727ee3b4f1445fe6d5e4133064ec2e4115069418a5bb6/cloudpickle-3.1.2-py3-none-any.whl.metadata
Using cached pyod-3.6.7-py3-none-any.whl (417 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 306.1/306.1 kB 2.4 MB/s eta 0:00:0013.0 MB/s eta 0:00:01
  Attempting uninstall: cloudpickle
    Found existing installation: cloudpickle 2.2.1
    Uninstalling cloudpickle-2.2.1:
      Successfully uni

In [22]:
import sys
sys.path.append("COPOD")

from models.cod import COPOD

print("COPOD imported successfully")

COPOD imported successfully


In [23]:
from sklearn.metrics import roc_auc_score, average_precision_score

outliers_fraction = y.sum() / len(y)

clf = COPOD(contamination=outliers_fraction)

clf.fit(X_train)

scores = clf.decision_function(X_test)

roc_auc = roc_auc_score(y_test, scores)
average_precision = average_precision_score(y_test, scores)

print("ROC-AUC:", roc_auc)
print("Average Precision:", average_precision)

ROC-AUC: 0.9925678838951311
Average Precision: 0.9848296214330675


In [24]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np

roc_scores = []
ap_scores = []

for i in range(10):

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.4,
        random_state=i
    )

    scaler = StandardScaler()
    X_train_norm = scaler.fit_transform(X_train)
    X_test_norm = scaler.transform(X_test)

    clf = COPOD(contamination=outliers_fraction)

    clf.fit(X_train_norm)

    test_scores = clf.decision_function(X_test_norm)

    roc = roc_auc_score(y_test, test_scores)
    ap = average_precision_score(y_test, test_scores)

    roc_scores.append(roc)
    ap_scores.append(ap)

    print(f"Iteration {i+1}: ROC-AUC = {roc:.4f}, AP = {ap:.4f}")

print("\nMean ROC-AUC:", np.mean(roc_scores))
print("Mean Average Precision:", np.mean(ap_scores))


Iteration 1: ROC-AUC = 0.9925, AP = 0.9862
Iteration 2: ROC-AUC = 0.9957, AP = 0.9914
Iteration 3: ROC-AUC = 0.9897, AP = 0.9819
Iteration 4: ROC-AUC = 0.9920, AP = 0.9847
Iteration 5: ROC-AUC = 0.9924, AP = 0.9866
Iteration 6: ROC-AUC = 0.9935, AP = 0.9871
Iteration 7: ROC-AUC = 0.9943, AP = 0.9863
Iteration 8: ROC-AUC = 0.9953, AP = 0.9913
Iteration 9: ROC-AUC = 0.9901, AP = 0.9820
Iteration 10: ROC-AUC = 0.9995, AP = 0.9992

Mean ROC-AUC: 0.9934983093381945
Mean Average Precision: 0.9876707650888837


In [25]:
paper_roc = 0.9936
your_roc = np.mean(roc_scores)

difference = your_roc - paper_roc

print("Paper ROC-AUC:", paper_roc)
print("Our ROC-AUC:", your_roc)
print("Difference:", difference)


Paper ROC-AUC: 0.9936
Our ROC-AUC: 0.9934983093381945
Difference: -0.00010169066180554243


In [26]:
import pandas as pd

results_table = pd.DataFrame({
    "Metric": ["ROC-AUC", "Average Precision"],
    "Paper": [0.9936, np.nan],
    "Reproduction": [np.mean(roc_scores), np.mean(ap_scores)]
})

results_table


,Metric,Paper,Reproduction
0,ROC-AUC,0.9936,0.993498
1,Average Precision,NaN,0.987671


In [27]:
results_table = pd.DataFrame({
    "Metric": ["ROC-AUC", "Average Precision"],
    "Paper": [0.9936, "Not reported"],
    "Reproduction": [np.mean(roc_scores), np.mean(ap_scores)]
})

results_table

,Metric,Paper,Reproduction
0,ROC-AUC,0.9936,0.993498
1,Average Precision,Not reported,0.987671


In [28]:
parameters = {
    "Dataset": "BreastW",
    "Total observations": 683,
    "Features": 9,
    "Training split": "60%",
    "Test split": "40%",
    "Number of runs": 10,
    "Random states": "0–9",
    "Standardization": "StandardScaler",
    "Algorithm": "COPOD",
    "Contamination": outliers_fraction,
    "Evaluation metrics": "ROC-AUC, Average Precision"
}

for key, value in parameters.items():
    print(f"{key}: {value}")

Dataset: BreastW
Total observations: 683
Features: 9
Training split: 60%
Test split: 40%
Number of runs: 10
Random states: 0–9
Standardization: StandardScaler
Algorithm: COPOD
Contamination: 0.34992679355783307
Evaluation metrics: ROC-AUC, Average Precision


In [29]:
roc_mean = np.mean(roc_scores)
roc_std = np.std(roc_scores, ddof=1)

ap_mean = np.mean(ap_scores)
ap_std = np.std(ap_scores, ddof=1)

paper_roc = 0.9936
difference = roc_mean - paper_roc
percentage_difference = (difference / paper_roc) * 100

print(f"Mean ROC-AUC: {roc_mean:.4f}")
print(f"ROC-AUC standard deviation: {roc_std:.4f}")
print(f"Mean Average Precision: {ap_mean:.4f}")
print(f"AP standard deviation: {ap_std:.4f}")
print(f"Difference from paper: {difference:.6f}")
print(f"Percentage difference: {percentage_difference:.4f}%")

Mean ROC-AUC: 0.9935
ROC-AUC standard deviation: 0.0029
Mean Average Precision: 0.9877
AP standard deviation: 0.0052
Difference from paper: -0.000102
Percentage difference: -0.0102%


In [30]:
print("ROC-AUC range:", min(roc_scores), "to", max(roc_scores))
print("AP range:", min(ap_scores), "to", max(ap_scores))

ROC-AUC range: 0.9896830825353397 to 0.9995402298850575
AP range: 0.9819143311638994 to 0.9991790335948834


In [31]:
final_results = pd.DataFrame({
    "Metric": ["ROC-AUC", "Average Precision"],
    "Paper": [0.9936, "Not reported"],
    "Reproduction": [roc_mean, ap_mean],
    "Std. Dev.": [roc_std, ap_std]
})

final_results


,Metric,Paper,Reproduction,Std. Dev.
0,ROC-AUC,0.9936,0.993498,0.002900
1,Average Precision,Not reported,0.987671,0.005165
